In [0]:
from pyspark.sql.functions import *

In [0]:
incremental_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/default/retail_data/sales_incremental.csv")
)

In [0]:
incremental_df = (
    incremental_df
    .withColumnRenamed("sls_ord_num", "order_number")
    .withColumnRenamed("sls_prd_key", "product_key")
    .withColumnRenamed("sls_cust_id", "customer_id")
    .withColumnRenamed("sls_order_dt", "order_date")
    .withColumnRenamed("sls_ship_dt", "ship_date")
    .withColumnRenamed("sls_due_dt", "due_date")
    .withColumnRenamed("sls_sales", "sales_amount")
    .withColumnRenamed("sls_quantity", "quantity")
    .withColumnRenamed("sls_price", "price")
)

display(incremental_df)

incremental_df.createOrReplaceTempView("incremental_sales")

order_number,product_key,customer_id,order_date,ship_date,due_date,sales_amount,quantity,price
SO999901,BK-R93R-44,25001,20140102,20140104,20140107,3200,2,1600
SO999902,BK-R93R-48,25002,20140105,20140107,20140110,1800,1,1800
SO999903,BK-M82S-44,25003,20140106,20140108,20140111,4100,2,2050
SO999904,BK-R50R-52,25004,20140108,20140110,20140113,2800,1,2800
SO43720,BK-R93R-44,13264,20140101,20140103,20140106,5000,2,2500
SO43734,BK-R93R-48,27604,20140102,20140104,20140107,4200,3,1400


In [0]:
spark.sql("""
MERGE INTO silver.sales AS target
USING incremental_sales AS source
ON target.order_number = source.order_number

WHEN MATCHED THEN
UPDATE SET
    target.product_key = source.product_key,
    target.customer_id = source.customer_id,
    target.order_date = source.order_date,
    target.ship_date = source.ship_date,
    target.due_date = source.due_date,
    target.sales_amount = source.sales_amount,
    target.quantity = source.quantity,
    target.price = source.price

WHEN NOT MATCHED THEN
INSERT (
    order_number,
    product_key,
    customer_id,
    order_date,
    ship_date,
    due_date,
    sales_amount,
    quantity,
    price
)
VALUES (
    source.order_number,
    source.product_key,
    source.customer_id,
    source.order_date,
    source.ship_date,
    source.due_date,
    source.sales_amount,
    source.quantity,
    source.price
)
""")



print("Incremental Load Completed Successfully")


print("Total Records After MERGE :",
      spark.table("silver.sales").count())

Incremental Load Completed Successfully
Total Records After MERGE : 60390


In [0]:
display(
    spark.sql("""
    SELECT *
    FROM silver.sales
    WHERE order_number IN (
        'SO999901',
        'SO999902',
        'SO999903',
        'SO999904'
    )
    ORDER BY order_number
    """)
)

order_number,product_key,customer_id,order_date,ship_date,due_date,sales_amount,quantity,price
SO999901,BK-R93R-44,25001,20140102,20140104,20140107,3200.0,2,1600.0
SO999902,BK-R93R-48,25002,20140105,20140107,20140110,1800.0,1,1800.0
SO999903,BK-M82S-44,25003,20140106,20140108,20140111,4100.0,2,2050.0
SO999904,BK-R50R-52,25004,20140108,20140110,20140113,2800.0,1,2800.0


In [0]:
display(spark.table("silver.sales"))

order_number,product_key,customer_id,order_date,ship_date,due_date,sales_amount,quantity,price
SO43750,BK-R93R-44,11591,20110111,20110118,20110123,3578.0,1,3578.0
SO43755,BK-R93R-44,27670,20110112,20110119,20110124,3578.0,1,3578.0
SO43760,BK-R93R-56,16352,20110113,20110120,20110125,3578.0,1,3578.0
SO43763,BK-R93R-52,16525,20110114,20110121,20110126,3578.0,1,3578.0
SO43769,BK-R93R-48,21659,20110116,20110123,20110128,3578.0,1,3578.0
SO43791,BK-R93R-56,16484,20110119,20110126,20110131,3578.0,1,3578.0
SO43792,BK-R93R-48,16623,20110119,20110126,20110131,3578.0,1,3578.0
SO43801,BK-R93R-44,13583,20110121,20110128,20110202,3578.0,1,3578.0
SO43805,BK-R50R-48,14510,20110122,20110129,20110203,699.0,1,699.0
SO43829,BK-R93R-62,27611,20110126,20110202,20110207,3578.0,1,3578.0
